In [1]:
!pip install fairlearn ucimlrepo folktables tqdm
import os, gc, json, time, copy, random, warnings, traceback
import numpy as np
import pandas as pd
from scipy import stats
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

WORK_DIR = "/kaggle/working"
PLOT_DIR = os.path.join(WORK_DIR, "agad_plots_fair")
os.makedirs(PLOT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEEDS_FINAL  = [42, 7, 123, 2024, 99]
RUN_DATASETS = ["adult", "acs_income", "acs_employment", "communities_crime"]

HIDDEN_DIM   = 128
DROPOUT      = 0.25
LR           = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE     = 35
GRL_MAX      = 1.0
ADV_STEPS    = 3

LAMBDA_MAX   = 6.0
MIN_SG_N     = 40
MIN_SG_FRAC  = 0.03
MAX_SG_FRAC  = 0.50
MAX_DEPTH    = 3
TOP_K_REPORT = 5

TOP_K_OPT           = 5
PER_SG_LAMBDA_MIN    = 0.25
PER_SG_LAMBDA_MAX    = 5.0

# --- NEW: size-weighting for the audit signal ---
# Raw per-subgroup gaps are noisy for small n (a 40-person subgroup can show a
# spuriously large TPR/FPR gap purely from sampling variance). We discount each
# subgroup's gap by a standard-error-style 1/sqrt(n) factor before ranking, so
# small subgroups need a much bigger observed gap to make it into the top-k
# that actually drives the adversarial fairness loss. This does NOT remove
# small subgroups from the audit universe (enumerate_subgroups / MIN_SG_N is
# unchanged) -- it just stops them from dominating top_subgroups/fg_k/lambda
# purely because they're volatile.
SIZE_WEIGHT_REF_N   = 500   # reference size: subgroups at this size get weight ~1.0
SIZE_WEIGHT_MIN     = 0.15  # floor so a tiny subgroup isn't zeroed out entirely
SIZE_WEIGHT_ENABLED = True

CONVERGENCE_THRESHOLD = 0.08
CONVERGENCE_PATIENCE  = 3
CURRICULUM_MIN_EPOCH_PHASE1 = 3
CURRICULUM_MIN_EPOCH_PHASE2 = 8

FULL_CFG = dict(
    adult             = dict(epochs=80,  batch=2048),
    acs_income        = dict(epochs=120, batch=4096),
    acs_employment    = dict(epochs=140, batch=4096),
    communities_crime = dict(epochs=130, batch=256),
)

VANILLA_ACC = dict(
    adult             = 0.7886,
    acs_income        = 0.7978,
    acs_employment    = 0.8075,
    communities_crime = 0.7118,
)
ACC_TOLERANCE = 0.05

# --- AUC floor removed from disqualification logic. We still *compute* and
# report AUC for every method/cell (it's diagnostically useful to see if
# ranking quality moved), but it no longer disqualifies a method from being
# considered "best" or "qualified". Only the accuracy floor gates that now.
VANILLA_AUC = dict(
    adult             = 0.85,
    acs_income         = 0.87,
    acs_employment     = 0.88,
    communities_crime  = 0.79,
)
AUC_TOLERANCE = 0.04
USE_AUC_FLOOR = False   # <-- flip to True to restore old AUC-disqualification behavior

FLOOR_PENALTY_WEIGHT = 17.5

PATIENCE_FINAL  = PATIENCE

BASELINE_ACC_FLOOR_TOLERANCE = ACC_TOLERANCE
BASELINE_AUC_FLOOR_TOLERANCE = AUC_TOLERANCE

PALETTE = {
    "vanilla"          : "#6c757d",
    "adv_eo"           : "#4e9af1",
    "adv_dp"           : "#1a6fc4",
    "fairlearn_eo"     : "#b07ded",
    "fairlearn_dp"     : "#7c3aed",
    "prejudice_rem_eo" : "#f9a8d4",
    "prejudice_rem_dp" : "#db2777",
    "agad_eo_tuned"    : "#2dc653",
    "agad_dp_tuned"    : "#1a7c34",
    "abl_no_auditor_eo": "#fde68a",
    "abl_no_auditor_dp": "#d97706",
}

METHOD_LABELS = {
    "vanilla"          : "Vanilla NN",
    "adv_eo"           : "Adv-EO (tuned)",
    "adv_dp"           : "Adv-DP (tuned)",
    "fairlearn_eo"     : "FL-AdvDeb-EO (tuned)",
    "fairlearn_dp"     : "FL-AdvDeb-DP (tuned)",
    "prejudice_rem_eo" : "PrejRem-EO (tuned)",
    "prejudice_rem_dp" : "PrejRem-DP (tuned)",
    "agad_eo_tuned"    : "AGAD-EO \u2605",
    "agad_dp_tuned"    : "AGAD-DP \u2605",
    "abl_no_auditor_eo": "No-Auditor-EO",
    "abl_no_auditor_dp": "No-Auditor-DP",
}

DS_LABELS = {
    "adult"            : "Adult Income",
    "acs_income"       : "ACS Income",
    "acs_employment"   : "ACS Employment",
    "communities_crime": "Communities & Crime",
}

COMPARISON_METHODS = [
    "vanilla", "adv_eo", "adv_dp",
    "fairlearn_eo", "fairlearn_dp",
    "prejudice_rem_eo", "prejudice_rem_dp",
    "agad_eo_tuned", "agad_dp_tuned",
]

ABLATION_EO_METHODS = ["agad_eo_tuned", "abl_no_auditor_eo"]
ABLATION_DP_METHODS = ["agad_dp_tuned", "abl_no_auditor_dp"]

ABLATION_LABELS = {
    "agad_eo_tuned"    : "Full AGAD\n(Auditor+GRL)",
    "agad_dp_tuned"    : "Full AGAD\n(Auditor+GRL)",
    "abl_no_auditor_eo": "GRL only\n(no Auditor)",
    "abl_no_auditor_dp": "GRL only\n(no Auditor)",
}

HARDCODED_PARAMS = {
    "adult": {
        "adv_eo": {"adv_weight": 3.939524089398576, "adv_lr_mult": 0.5521962529380021},
        "adv_dp": {"adv_weight": 3.629486157785268, "adv_lr_mult": 0.9074967586680978},
        "fairlearn_eo": {"learning_rate": 0.0013917801405643393},
        "fairlearn_dp": {"learning_rate": 0.0007054952283772943},
        "prejudice_rem_eo": {"eta": 10.783257042210106},
        "prejudice_rem_dp": {"eta": 29.991939146566686},
        "agad_eo": {"lambda0": 2.7903173783975794, "alpha": 12.348203143313157,
                    "task_weight": 1.2544188056074093, "vt_smooth": 1,
                    "acc_penalty_coef": 1.700301474829005},
        "agad_dp": {"lambda0": 0.9831530896992369, "alpha": 10.518518327983717,
                    "task_weight": 1.5706804950409188, "vt_smooth": 1,
                    "acc_penalty_coef": 0.28461083381155255},
    },
    "acs_income": {
        "adv_eo": {"adv_weight": 3.9896949423303694, "adv_lr_mult": 2.1870038830112555},
        "adv_dp": {"adv_weight": 3.9497307520418077, "adv_lr_mult": 2.9761087754404008},
        "fairlearn_eo": {"learning_rate": 0.0010724664962240846},
        "fairlearn_dp": {"learning_rate": 0.0026490945811978926},
        "prejudice_rem_eo": {"eta": 0.5004003176479138},
        "prejudice_rem_dp": {"eta": 29.868949656942952},
        "agad_eo": {"lambda0": 3.705413140111432, "alpha": 15.004341396060491,
                    "task_weight": 1.2933607150054032, "vt_smooth": 1,
                    "acc_penalty_coef": 0.20827535875610745},
        "agad_dp": {"lambda0": 2.18971496634526, "alpha": 14.047215720261054,
                    "task_weight": 1.674392820737905, "vt_smooth": 1,
                    "acc_penalty_coef": 0.35196075124030285},
    },
    "acs_employment": {
        "adv_eo": {"adv_weight": 3.6495235935824275, "adv_lr_mult": 2.3933899462470904},
        "adv_dp": {"adv_weight": 3.996338146550311, "adv_lr_mult": 2.990869079783399},
        "fairlearn_eo": {"learning_rate": 0.0006092374471534239},
        "fairlearn_dp": {"learning_rate": 0.00038181045555141485},
        "prejudice_rem_eo": {"eta": 0.5017191430137917},
        "prejudice_rem_dp": {"eta": 27.903549992531882},
        "agad_eo": {"lambda0": 4.643578760014326, "alpha": 16.583294286203877,
                    "task_weight": 1.4134296830610005, "vt_smooth": 2,
                    "acc_penalty_coef": 1.9791617962235093},
        "agad_dp": {"lambda0": 1.5345177366513445, "alpha": 25.353507403686336,
                    "task_weight": 1.1404381604692637, "vt_smooth": 1,
                    "acc_penalty_coef": 1.4901164170799803},
    },
    "communities_crime": {
        "adv_eo": {"adv_weight": 3.9557488620121397, "adv_lr_mult": 2.3320306966812323},
        "adv_dp": {"adv_weight": 3.976804715167002, "adv_lr_mult": 2.535068661529437},
        "fairlearn_eo": {"learning_rate": 0.0039911308506321574},
        "fairlearn_dp": {"learning_rate": 0.0009334344135812729},
        "prejudice_rem_eo": {"eta": 0.5268048775667181},
        "prejudice_rem_dp": {"eta": 26.47324407106837},
        "agad_eo": {"lambda0": 4.948945666336031, "alpha": 23.99078304388506,
                    "task_weight": 1.0078681614886604, "vt_smooth": 1,
                    "acc_penalty_coef": 2.973645521148309},
        "agad_dp": {"lambda0": 3.4564766334115795, "alpha": 32.19572760950875,
                    "task_weight": 1.0162160196762846, "vt_smooth": 2,
                    "acc_penalty_coef": 1.071652695224908},
    },
}

print("=" * 70)
print("  AGAD v9 -- hardcoded hyperparams, fg_k checkpointing, fairlearn nan fix,")
print("             size-weighted audit, AUC floor removed, verbose + tqdm")
print("=" * 70)
print(f"  Device       : {DEVICE}")
print(f"  Seeds        : {SEEDS_FINAL}")
print(f"  TOP_K_OPT    : {TOP_K_OPT}")
print(f"  Per-sg lambda: [{PER_SG_LAMBDA_MIN}, {PER_SG_LAMBDA_MAX}]")
print(f"  Size weight  : enabled={SIZE_WEIGHT_ENABLED}  ref_n={SIZE_WEIGHT_REF_N}  min={SIZE_WEIGHT_MIN}")
print(f"  AUC floor    : enabled={USE_AUC_FLOOR} (disqualification now driven by acc floor only)")
print(f"  Curriculum   : convergence-based (threshold={CONVERGENCE_THRESHOLD}, "
      f"patience={CONVERGENCE_PATIENCE})")


def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def ensure_binary(sv):
    sv = np.asarray(sv).ravel(); u = np.unique(sv)
    if len(u) <= 1: return np.zeros(len(sv), int)
    if len(u) == 2: return (sv == u[1]).astype(int)
    maj = u[np.argmax([(sv == v).sum() for v in u])]
    return (sv != maj).astype(int)

def safe_auc(yt, yp):
    try:    return float(roc_auc_score(yt, yp)) if len(np.unique(yt)) >= 2 else float("nan")
    except: return float("nan")

def strat_split(X, y, sens_list, test_size=0.2, seed=42):
    key = np.array(y).astype(str)
    for s in sens_list:
        key = key + "_" + np.array(s).astype(str)
    u, c = np.unique(key, return_counts=True)
    key[np.isin(key, u[c < 2])] = np.array(y)[np.isin(key, u[c < 2])].astype(str)
    try:
        return train_test_split(np.arange(len(y)), test_size=test_size,
                                stratify=key, random_state=seed)
    except:
        return train_test_split(np.arange(len(y)), test_size=test_size,
                                stratify=y, random_state=seed)

def eo_gap(y_true, y_prob, mask):
    sg_t = y_true[mask]; sg_p = y_prob[mask]
    ot_t = y_true[~mask]; ot_p = y_prob[~mask]
    if len(np.unique(sg_t)) < 2 or len(np.unique(ot_t)) < 2: return 0.0
    tpr_sg = sg_p[sg_t == 1].mean() if (sg_t == 1).sum() > 0 else 0.0
    tpr_ot = ot_p[ot_t == 1].mean() if (ot_t == 1).sum() > 0 else 0.0
    fpr_sg = sg_p[sg_t == 0].mean() if (sg_t == 0).sum() > 0 else 0.0
    fpr_ot = ot_p[ot_t == 0].mean() if (ot_t == 0).sum() > 0 else 0.0
    return float(max(abs(tpr_sg - tpr_ot), abs(fpr_sg - fpr_ot)))

def dp_gap(y_prob, mask):
    return float(abs(y_prob[mask].mean() - y_prob.mean()))

def size_weight(n, ref_n=SIZE_WEIGHT_REF_N, min_w=SIZE_WEIGHT_MIN):
    """Standard-error-style discount: weight ~ sqrt(n)/sqrt(ref_n), capped at 1.0
    and floored at min_w. A subgroup at ref_n gets weight 1.0; a subgroup at
    n/4 of ref_n gets weight 0.5; very small subgroups are heavily discounted
    but never fully zeroed (min_w keeps them in play if their gap is huge)."""
    if not SIZE_WEIGHT_ENABLED:
        return 1.0
    w = np.sqrt(max(n, 1) / float(ref_n))
    return float(np.clip(w, min_w, 1.0))

def fg_metric(y_true, y_prob, sgs, k=TOP_K_REPORT, metric="eo"):
    gaps = []
    for sg in sgs:
        mem = sg["mem"]
        if len(np.unique(y_true[mem])) < 2: continue
        g = eo_gap(y_true, y_prob, mem) if metric == "eo" else dp_gap(y_prob, mem)
        gaps.append(g)
    if not gaps: return 0.0
    gaps.sort(reverse=True)
    return float(np.mean(gaps[:k]))

def enumerate_subgroups(sv_bin_list, attr_names, n, max_depth=2):
    n_attr = len(attr_names)
    min_n  = max(MIN_SG_N, int(MIN_SG_FRAC * n))
    max_n  = int(MAX_SG_FRAC * n)
    sgs, seen = [], set()
    for mask in range(1, 2 ** n_attr):
        active = [i for i in range(n_attr) if mask & (1 << i)]
        if len(active) > max_depth: continue
        for vals in range(2 ** len(active)):
            req = [(vals >> j) & 1 for j in range(len(active))]
            mem = np.ones(n, dtype=bool)
            for ai, rv in zip(active, req):
                mem &= (sv_bin_list[ai] == rv)
            key = mem.tobytes()
            if key in seen: continue
            seen.add(key)
            sz = int(mem.sum())
            if sz < min_n or sz > max_n: continue
            spec = [(attr_names[i], req[j]) for j, i in enumerate(active)]
            sgs.append({"mem": mem, "spec": spec, "active": tuple(active), "req": tuple(req)})
    return sgs

def audit(depth, y_val, p_val, sv_val_bin, attr_names, metric="eo"):
    """Ranks subgroups by a SIZE-WEIGHTED gap: weighted_gap = gap * size_weight(n).
    This means a small, noisy subgroup with a large raw gap gets discounted
    before ranking, so it needs a much bigger gap to reach the top-k that
    drives the adversarial loss and fg_k checkpoint score. The raw (unweighted)
    gap is still returned per-subgroup for reporting/analysis.
    """
    depth  = min(depth, MAX_DEPTH)
    n      = len(p_val)
    sgs    = enumerate_subgroups(sv_val_bin, attr_names, n, max_depth=depth)
    ranked = []
    for sg in sgs:
        mem = sg["mem"]
        if len(np.unique(y_val[mem])) < 2: continue
        raw_gap = eo_gap(y_val, p_val, mem) if metric == "eo" else dp_gap(p_val, mem)
        sg_n    = int(mem.sum())
        w       = size_weight(sg_n)
        w_gap   = raw_gap * w
        ranked.append({"gap": raw_gap, "weighted_gap": w_gap, "n": sg_n, "weight": w,
                        "spec": sg["spec"], "mem": mem,
                        "active": sg["active"], "req": sg["req"]})
    # rank by weighted gap (size-discounted), not raw gap
    ranked.sort(key=lambda x: -x["weighted_gap"])
    worst_gap  = ranked[0]["gap"] if ranked else 0.0          # raw, for reporting (unchanged semantics)
    worst_spec = ranked[0]["spec"] if ranked else None
    topk_specs = [r["spec"] for r in ranked[:6]]
    # fg_k is computed on the size-weighted ranking's top-k RAW gaps (consistent
    # with "these are the k most credible worst-off subgroups", not the k
    # noisiest small ones)
    fg_k       = float(np.mean([r["gap"] for r in ranked[:TOP_K_REPORT]])) if ranked else 0.0
    return float(worst_gap), worst_spec, float(fg_k), topk_specs, ranked

def adaptive_lambda(V_t, lambda0, alpha):
    return min(lambda0 * (1.0 + alpha * V_t), LAMBDA_MAX)

def build_intersection_labels(sv_bin_list, n_attrs):
    n      = len(sv_bin_list[0])
    labels = np.zeros(n, dtype=np.int64)
    for sv in sv_bin_list:
        labels = labels * 2 + np.asarray(sv, dtype=np.int64)
    return labels

def evaluate(proba, y, sv_bin_list, attr_names, tag="", verbose=True):
    proba = np.clip(np.nan_to_num(proba, nan=0.5, posinf=1.0, neginf=0.0), 1e-7, 1 - 1e-7)
    pred  = (proba > 0.5).astype(int)
    acc   = float(accuracy_score(y, pred))
    auc   = safe_auc(y, proba)
    n     = len(proba)
    sgs   = enumerate_subgroups(sv_bin_list, attr_names, n, max_depth=MAX_DEPTH)
    wc_eo = max((eo_gap(y, proba, sg["mem"])
                 for sg in sgs if len(np.unique(y[sg["mem"]])) >= 2), default=0.0)
    wc_dp = max((dp_gap(proba, sg["mem"]) for sg in sgs), default=0.0)
    fg_eo = fg_metric(y, proba, sgs, k=TOP_K_REPORT, metric="eo")
    fg_dp = fg_metric(y, proba, sgs, k=TOP_K_REPORT, metric="dp")
    if verbose:
        print(f"    [{tag:<36}]  WC-EO={wc_eo:.4f}  WC-DP={wc_dp:.4f}  "
              f"FG-EO={fg_eo:.4f}  FG-DP={fg_dp:.4f}  Acc={acc:.4f}  AUC={auc:.4f}")
    return {"wc_eo": wc_eo, "wc_dp": wc_dp, "fg_eo": fg_eo,
            "fg_dp": fg_dp, "acc": acc, "auc": auc}

def aggregate_seeds(results_list, tag_for_warning=""):
    keys = ["wc_eo", "wc_dp", "fg_eo", "fg_dp", "acc", "auc"]
    agg  = {}
    n_total = len(results_list)
    n_failed = sum(
        1 for r in results_list
        if r.get("acc") is None or (isinstance(r.get("acc"), float) and np.isnan(r.get("acc")))
    )
    for k in keys:
        vals = [r[k] for r in results_list if k in r and r[k] is not None
                and not (isinstance(r[k], float) and np.isnan(r[k]))]
        agg[k]          = float(np.mean(vals)) if vals else float("nan")
        agg[k + "_std"] = float(np.std(vals))  if len(vals) > 1 else 0.0
        agg[k + "_all"] = [float(v) for v in vals]
    agg["n_seeds_total"]  = n_total
    agg["n_seeds_failed"] = n_failed
    if n_failed > 0:
        print(f"    [WARNING] {tag_for_warning}: {n_failed}/{n_total} seeds failed "
              f"(NaN result) -- aggregated stats computed over surviving seeds only")
    return agg

def pareto_frontier(results_dict, fairness_key, acc_key):
    methods = list(results_dict.keys())
    frontier = []
    for m in methods:
        f_m   = results_dict[m].get(fairness_key, float("nan"))
        acc_m = results_dict[m].get(acc_key, float("nan"))
        if np.isnan(f_m) or np.isnan(acc_m):
            continue
        dominated = False
        for m2 in methods:
            if m2 == m: continue
            f_m2   = results_dict[m2].get(fairness_key, float("nan"))
            acc_m2 = results_dict[m2].get(acc_key, float("nan"))
            if np.isnan(f_m2) or np.isnan(acc_m2):
                continue
            if f_m2 <= f_m and acc_m2 >= acc_m and (f_m2 < f_m or acc_m2 > acc_m):
                dominated = True
                break
        if not dominated:
            frontier.append(m)
    return frontier

def print_section(title, width=70):
    print(); print("=" * width); print(f"  {title}"); print("=" * width)

def print_subsection(title, width=70):
    print(); print("-" * width); print(f"  {title}"); print("-" * width)


class GRL(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha; return x.clone()
    @staticmethod
    def backward(ctx, g):
        return -ctx.alpha * g, None

class GradReversal(nn.Module):
    def __init__(self, alpha=1.0):
        super().__init__(); self.alpha = alpha
    def forward(self, x):
        return GRL.apply(x, self.alpha)

class Encoder(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        h = HIDDEN_DIM
        self.rep_dim = h // 2 + 32
        self.net = nn.Sequential(
            nn.Linear(in_dim, h),         nn.BatchNorm1d(h),            nn.GELU(), nn.Dropout(DROPOUT),
            nn.Linear(h, 256),            nn.BatchNorm1d(256),          nn.GELU(), nn.Dropout(DROPOUT),
            nn.Linear(256, 128),          nn.BatchNorm1d(128),          nn.GELU(), nn.Dropout(DROPOUT * 0.5),
            nn.Linear(128, self.rep_dim), nn.BatchNorm1d(self.rep_dim), nn.GELU(), nn.Dropout(DROPOUT * 0.5),
        )
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight); nn.init.zeros_(m.bias)
    def forward(self, x): return self.net(x)

class PredictorHead(nn.Module):
    def __init__(self, rep_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(rep_dim, rep_dim // 2), nn.GELU(),
            nn.Linear(rep_dim // 2, 1),
        )
    def forward(self, z): return self.net(z).squeeze(1)

class IntersectionAdversaryHead(nn.Module):
    def __init__(self, rep_dim, n_marginal_attrs, n_intersection_classes):
        super().__init__()
        h = HIDDEN_DIM // 2
        self.grl   = GradReversal(1.0)
        self.trunk = nn.Sequential(
            nn.Linear(rep_dim, h), nn.GELU(), nn.Dropout(DROPOUT),
            nn.Linear(h, h // 2), nn.GELU(),
        )
        self.marginal_out      = nn.Linear(h // 2, n_marginal_attrs)
        self.intersection_out  = nn.Linear(h // 2, n_intersection_classes)
        self.violation_out     = nn.Linear(h // 2, 1)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight); nn.init.zeros_(m.bias)

    def set_alpha(self, a): self.grl.alpha = a

    def forward(self, z, alpha=None):
        if alpha is not None: self.grl.alpha = alpha
        h = self.trunk(self.grl(z))
        return self.marginal_out(h), self.intersection_out(h), self.violation_out(h).squeeze(1)


def run_agad_with_hparams(d, cfg, metric, hp, seed=42,
                           verbose=False, epoch_verbose=False,
                           fg_ckpt_weight=1.0,
                           min_ckpt_epoch=0,
                           disable_auditor=False,
                           patience=PATIENCE,
                           show_progress=True,
                           progress_desc=""):
    set_seed(seed)

    lambda0     = hp["lambda0"]
    alpha_val   = hp["alpha"]
    task_weight = hp["task_weight"]
    vt_smooth   = hp.get("vt_smooth", 1)
    if "fg_ckpt_weight" in hp:
        fg_ckpt_weight = hp["fg_ckpt_weight"]
    acc_penalty_coef = hp.get("acc_penalty_coef", 0.5)

    n_attrs = len(d["attr_names"])
    n_inter = 2 ** n_attrs

    Xt   = torch.tensor(d["X_tr"],  dtype=torch.float32, device=DEVICE)
    yt   = torch.tensor(d["y_tr"],  dtype=torch.float32, device=DEVICE)
    Xv   = torch.tensor(d["X_val"], dtype=torch.float32, device=DEVICE)
    Xte  = torch.tensor(d["X_te"],  dtype=torch.float32, device=DEVICE)
    sv_t = [torch.tensor(sv, dtype=torch.float32, device=DEVICE) for sv in d["sv_tr"]]
    inter_labels_np = build_intersection_labels(d["sv_tr"], n_attrs)
    inter_t  = torch.tensor(inter_labels_np, dtype=torch.long, device=DEVICE)
    idx_t    = torch.arange(len(d["y_tr"]), dtype=torch.long)

    enc  = Encoder(d["X_tr"].shape[1]).to(DEVICE)
    head = PredictorHead(enc.rep_dim).to(DEVICE)
    adv  = IntersectionAdversaryHead(enc.rep_dim, n_attrs, n_inter).to(DEVICE)
    bce  = nn.BCEWithLogitsLoss()
    ce   = nn.CrossEntropyLoss()
    eps  = torch.tensor(1e-4, device=DEVICE)

    opt_enc = optim.AdamW(
        list(enc.parameters()) + list(head.parameters()),
        lr=LR, weight_decay=WEIGHT_DECAY)
    opt_adv = optim.AdamW(adv.parameters(),
                          lr=LR * 2.0, weight_decay=WEIGHT_DECAY)
    sched  = optim.lr_scheduler.CosineAnnealingLR(
        opt_enc, cfg["epochs"], eta_min=LR * 0.01)
    loader = DataLoader(
        TensorDataset(Xt, yt, idx_t),
        batch_size=cfg["batch"], shuffle=True, drop_last=True)

    best_score  = np.inf
    best_enc    = best_head = None
    best_epoch  = -1
    no_imp      = 0
    V_t         = 0.0
    fg_k        = 0.0
    acc_floor   = VANILLA_ACC[d["ds_key"]] - ACC_TOLERANCE
    vt_history  = []

    depth_limit          = 1
    depth1_conv_streak    = 0
    depth2_conv_streak    = 0

    per_sg_lambda = {}

    def sg_key(sg):
        return (sg["active"], sg["req"])

    def set_sg_lambda(key, V_i):
        lam = min(max(lambda0 * (1.0 + alpha_val * V_i), PER_SG_LAMBDA_MIN), PER_SG_LAMBDA_MAX)
        per_sg_lambda[key] = lam
        return lam

    top_subgroups = []

    def project_subgroup_to_train(sg, sv_tr_bin_list, n_train):
        mem = np.ones(n_train, dtype=bool)
        for ai, rv in zip(sg["active"], sg["req"]):
            mem &= (sv_tr_bin_list[ai] == rv)
        return mem

    epoch_iter = range(cfg["epochs"])
    pbar = tqdm(epoch_iter, desc=progress_desc, leave=False, disable=not show_progress)

    for epoch in pbar:
        enc.eval(); head.eval()
        with torch.no_grad():
            pv_scan = torch.sigmoid(head(enc(Xv))).cpu().numpy()

        if disable_auditor:
            V_t   = 0.0
            lam_t = lambda0
            fg_k  = 0.0
            top_subgroups = []
        else:
            V_raw, worst_spec, fg_k, topk_specs, ranked = audit(
                depth_limit, d["y_val"], pv_scan, d["sv_val"], d["attr_names"], metric=metric)
            vt_history.append(V_raw)
            if len(vt_history) > vt_smooth:
                vt_history.pop(0)
            V_t = float(np.mean(vt_history))
            lam_t = adaptive_lambda(V_t, lambda0, alpha_val)

            if depth_limit == 1:
                if V_raw < CONVERGENCE_THRESHOLD:
                    depth1_conv_streak += 1
                else:
                    depth1_conv_streak = 0
                if (depth1_conv_streak >= CONVERGENCE_PATIENCE
                        and epoch >= CURRICULUM_MIN_EPOCH_PHASE1
                        and MAX_DEPTH >= 2):
                    depth_limit = 2
            elif depth_limit == 2:
                if V_raw < CONVERGENCE_THRESHOLD:
                    depth2_conv_streak += 1
                else:
                    depth2_conv_streak = 0
                if (depth2_conv_streak >= CONVERGENCE_PATIENCE
                        and epoch >= CURRICULUM_MIN_EPOCH_PHASE2
                        and MAX_DEPTH >= 3):
                    depth_limit = 3

            # ranked is already sorted by weighted_gap (size-discounted) --
            # taking the head of this list is the size-aware top-k selection
            top_ranked = ranked[:TOP_K_OPT]
            n_train = len(d["y_tr"])

            forced_marginal_sgs = []
            for ai in range(n_attrs):
                mem_val = (np.asarray(d["sv_val"][ai]) == 1)
                if mem_val.sum() < 2 or (~mem_val).sum() < 2:
                    continue
                if metric == "eo":
                    gap = eo_gap(d["y_val"], pv_scan, mem_val)
                else:
                    gap = dp_gap(pv_scan, mem_val)
                forced_marginal_sgs.append({"gap": gap, "active": (ai,), "req": (1,)})

            combined = {}
            for sg in top_ranked:
                combined[sg_key(sg)] = sg
            for sg in forced_marginal_sgs:
                k = (sg["active"], sg["req"])
                if k not in combined:
                    combined[k] = sg

            top_subgroups = []
            for sg in combined.values():
                key = (sg["active"], sg["req"])
                lam_i = set_sg_lambda(key, sg["gap"])
                mem_tr = project_subgroup_to_train(sg, d["sv_tr"], n_train)
                if mem_tr.sum() < 5:
                    continue
                top_subgroups.append({
                    "mem_tr": mem_tr,
                    "lambda": lam_i,
                    "gap": sg["gap"],
                })

        grl_alpha = GRL_MAX * max(0.1, epoch / max(1, cfg["epochs"]))
        adv.set_alpha(grl_alpha)
        enc.train(); head.train(); adv.train()

        sg_masks_t   = []
        sg_lambdas   = []
        violation_target_np = np.zeros(len(d["y_tr"]), dtype=np.float32)
        for sg in top_subgroups:
            sg_masks_t.append(torch.tensor(sg["mem_tr"], device=DEVICE))
            sg_lambdas.append(sg["lambda"])
            violation_target_np[sg["mem_tr"]] = np.maximum(
                violation_target_np[sg["mem_tr"]], sg["gap"])
        violation_target_full = torch.tensor(violation_target_np, device=DEVICE)

        for xb, yb, bidx in loader:
            z_d = enc(xb).detach()
            vio_tgt_b = violation_target_full[bidx]
            for _ in range(ADV_STEPS):
                opt_adv.zero_grad(set_to_none=True)
                m_out, i_out, v_out = adv(z_d, alpha=0)
                loss_a = (
                    sum(nn.functional.binary_cross_entropy_with_logits(
                        m_out[:, i], sv_t[i][bidx])
                        for i in range(n_attrs)) / n_attrs
                    + ce(i_out, inter_t[bidx])
                    + nn.functional.mse_loss(v_out, vio_tgt_b))
                loss_a.backward()
                nn.utils.clip_grad_norm_(adv.parameters(), 1.0)
                opt_adv.step()

            opt_enc.zero_grad(set_to_none=True)
            z      = enc(xb)
            logits = head(z)
            prob   = torch.sigmoid(logits)
            task_loss = bce(logits, yb)

            fair_loss_global = torch.tensor(0.0, device=DEVICE)
            m_adv, i_adv, v_adv = adv(z)
            for i, sv in enumerate(sv_t):
                tgt = sv[bidx]
                fair_loss_global += nn.functional.binary_cross_entropy_with_logits(
                    m_adv[:, i], tgt)
            fair_loss_global += ce(i_adv, inter_t[bidx])
            fair_loss_global += nn.functional.mse_loss(v_adv, vio_tgt_b)
            fair_loss_global = fair_loss_global / max(n_attrs, 1)

            fair_loss_topk = torch.tensor(0.0, device=DEVICE)
            n_active_sg_terms = 0
            if sg_masks_t:
                yf_full = yb.float()
                for mask_full, lam_i in zip(sg_masks_t, sg_lambdas):
                    mem_b = mask_full[bidx]
                    if mem_b.sum() < 2:
                        continue
                    if metric == "eo":
                        yf = yf_full
                        in_g_y1 = mem_b.float() * yf
                        in_g_y0 = mem_b.float() * (1 - yf)
                        out_g_y1 = (~mem_b).float() * yf
                        out_g_y0 = (~mem_b).float() * (1 - yf)
                        if (in_g_y1.sum() > 1e-6 and out_g_y1.sum() > 1e-6
                                and in_g_y0.sum() > 1e-6 and out_g_y0.sum() > 1e-6):
                            tpr_in  = (prob * in_g_y1).sum()  / (in_g_y1.sum()  + eps)
                            tpr_out = (prob * out_g_y1).sum() / (out_g_y1.sum() + eps)
                            fpr_in  = (prob * in_g_y0).sum()  / (in_g_y0.sum()  + eps)
                            fpr_out = (prob * out_g_y0).sum() / (out_g_y0.sum() + eps)
                            sg_loss = torch.max(torch.abs(tpr_in - tpr_out),
                                                 torch.abs(fpr_in - fpr_out))
                            fair_loss_topk += lam_i * sg_loss
                            n_active_sg_terms += 1
                    else:
                        n_in  = mem_b.float().sum() + eps
                        n_out = (~mem_b).float().sum() + eps
                        p_in  = (prob * mem_b.float()).sum() / n_in
                        p_out = (prob * (~mem_b).float()).sum() / n_out
                        sg_loss = torch.abs(p_in - p_out)
                        fair_loss_topk += lam_i * sg_loss
                        n_active_sg_terms += 1
            if n_active_sg_terms > 0:
                fair_loss_topk = fair_loss_topk / n_active_sg_terms

            loss = (task_weight * task_loss
                    + lam_t * fair_loss_global
                    + fair_loss_topk)

            loss.backward()
            nn.utils.clip_grad_norm_(
                list(enc.parameters()) + list(head.parameters()), 0.5)
            opt_enc.step()

        sched.step()

        enc.eval(); head.eval()
        with torch.no_grad():
            pv  = torch.sigmoid(head(enc(Xv))).cpu().numpy()
        acc = accuracy_score(d["y_val"], (pv > 0.5).astype(int))

        floor_penalty = acc_penalty_coef * max(0.0, acc_floor - acc)
        acc_tiebreak  = -1e-4 * acc
        score = (fg_ckpt_weight * fg_k
                 + floor_penalty
                 + acc_tiebreak)

        if score < best_score and epoch >= min_ckpt_epoch:
            best_score = score
            best_enc   = copy.deepcopy(enc.state_dict())
            best_head  = copy.deepcopy(head.state_dict())
            best_epoch = epoch
            no_imp     = 0
        elif epoch >= min_ckpt_epoch:
            no_imp += 1

        if epoch < min_ckpt_epoch:
            best_enc  = copy.deepcopy(enc.state_dict())
            best_head = copy.deepcopy(head.state_dict())
            best_epoch = epoch

        pbar.set_postfix({
            "depth": depth_limit, "fg_k": f"{fg_k:.4f}", "lam": f"{lam_t:.2f}",
            "acc": f"{acc:.4f}", "best_ep": best_epoch, "no_imp": no_imp,
        })

        if epoch_verbose and (epoch % 20 == 0 or epoch == cfg["epochs"] - 1):
            print(f"      epoch {epoch:03d}  depth={depth_limit}  V_t={V_t:.4f}  fg_k={fg_k:.4f}  lam={lam_t:.3f}  "
                  f"acc={acc:.4f}  score={score:.5f}  no_imp={no_imp}  "
                  f"topk_sgs={len(top_subgroups)}")

        if no_imp >= patience:
            if epoch_verbose:
                print(f"      [early stop at epoch {epoch}]")
            break

    pbar.close()

    enc.load_state_dict(best_enc)
    head.load_state_dict(best_head)
    enc.eval(); head.eval()
    with torch.no_grad():
        proba = torch.sigmoid(head(enc(Xte))).cpu().numpy()

    tag = f"AGAD-{metric.upper()} (final)"
    result = evaluate(proba, d["y_te"], d["sv_te"], d["attr_names"],
                    tag=tag, verbose=verbose)
    result["best_epoch"] = best_epoch
    result["stopped_epoch"] = epoch
    return result


def run_vanilla(d, cfg, seed=42, show_progress=True, progress_desc=""):
    set_seed(seed)
    Xt  = torch.tensor(d["X_tr"],  dtype=torch.float32, device=DEVICE)
    yt  = torch.tensor(d["y_tr"],  dtype=torch.float32, device=DEVICE)
    Xv  = torch.tensor(d["X_val"], dtype=torch.float32, device=DEVICE)
    Xte = torch.tensor(d["X_te"],  dtype=torch.float32, device=DEVICE)
    enc  = Encoder(d["X_tr"].shape[1]).to(DEVICE)
    head = PredictorHead(enc.rep_dim).to(DEVICE)
    bce  = nn.BCEWithLogitsLoss()
    opt  = optim.AdamW(list(enc.parameters()) + list(head.parameters()),
                       lr=LR, weight_decay=WEIGHT_DECAY)
    sched  = optim.lr_scheduler.CosineAnnealingLR(opt, cfg["epochs"], eta_min=LR * 0.01)
    loader = DataLoader(TensorDataset(Xt, yt),
                        batch_size=cfg["batch"], shuffle=True, drop_last=True)
    best_score = np.inf; best_enc = best_head = None; no_imp = 0
    pbar = tqdm(range(cfg["epochs"]), desc=progress_desc, leave=False, disable=not show_progress)
    for epoch in pbar:
        enc.train(); head.train()
        for xb, yb in loader:
            opt.zero_grad(set_to_none=True)
            bce(head(enc(xb)), yb).backward()
            opt.step()
        sched.step()
        enc.eval(); head.eval()
        with torch.no_grad():
            pv = torch.sigmoid(head(enc(Xv))).cpu().numpy()
        auc = safe_auc(d["y_val"], pv)
        V, _, _, _, _ = audit(1, d["y_val"], pv, d["sv_val"], d["attr_names"])
        score = V + 0.12 * (1 - auc if not np.isnan(auc) else 0)
        if score < best_score:
            best_score = score; best_enc = copy.deepcopy(enc.state_dict())
            best_head  = copy.deepcopy(head.state_dict()); no_imp = 0
        else:
            no_imp += 1
        pbar.set_postfix({"auc": f"{auc:.4f}", "V": f"{V:.4f}", "no_imp": no_imp})
        if no_imp >= PATIENCE: break
    pbar.close()
    enc.load_state_dict(best_enc); head.load_state_dict(best_head)
    enc.eval(); head.eval()
    with torch.no_grad():
        proba = torch.sigmoid(head(enc(Xte))).cpu().numpy()
    return evaluate(proba, d["y_te"], d["sv_te"], d["attr_names"], tag="Vanilla NN", verbose=False)


def _adv_static(d, cfg, metric, seed=42, adv_weight=1.0, adv_lr_mult=1.0,
                 patience=PATIENCE, show_progress=True, progress_desc=""):
    set_seed(seed)
    n_attrs = len(d["attr_names"])
    Xt   = torch.tensor(d["X_tr"],  dtype=torch.float32, device=DEVICE)
    yt   = torch.tensor(d["y_tr"],  dtype=torch.float32, device=DEVICE)
    Xv   = torch.tensor(d["X_val"], dtype=torch.float32, device=DEVICE)
    Xte  = torch.tensor(d["X_te"],  dtype=torch.float32, device=DEVICE)
    sv_t = [torch.tensor(sv, dtype=torch.float32, device=DEVICE) for sv in d["sv_tr"]]
    idx_t = torch.arange(len(d["y_tr"]), dtype=torch.long)

    enc  = Encoder(d["X_tr"].shape[1]).to(DEVICE)
    head = PredictorHead(enc.rep_dim).to(DEVICE)

    adv_heads = nn.ModuleList([
        nn.Sequential(
            GradReversal(1.0),
            nn.Linear(enc.rep_dim, HIDDEN_DIM // 2), nn.ReLU(),
            nn.Linear(HIDDEN_DIM // 2, 1),
        ).to(DEVICE)
        for _ in range(n_attrs)
    ])

    bce = nn.BCEWithLogitsLoss()
    opt = optim.AdamW(
        list(enc.parameters()) + list(head.parameters()),
        lr=LR, weight_decay=WEIGHT_DECAY)
    opt_adv = optim.AdamW(adv_heads.parameters(),
                           lr=LR * adv_lr_mult, weight_decay=WEIGHT_DECAY)
    sched  = optim.lr_scheduler.CosineAnnealingLR(opt, cfg["epochs"], eta_min=LR * 0.01)
    loader = DataLoader(TensorDataset(Xt, yt, idx_t),
                        batch_size=cfg["batch"], shuffle=True, drop_last=True)

    best_score = np.inf; best_enc = best_head = None; no_imp = 0

    pbar = tqdm(range(cfg["epochs"]), desc=progress_desc, leave=False, disable=not show_progress)
    for epoch in pbar:
        alpha = GRL_MAX * max(0.1, epoch / max(1, cfg["epochs"]))
        for ah in adv_heads:
            ah[0].alpha = alpha

        enc.train(); head.train()
        for ah in adv_heads: ah.train()

        for xb, yb, bidx in loader:
            opt.zero_grad(set_to_none=True)
            opt_adv.zero_grad(set_to_none=True)
            z      = enc(xb)
            logits = head(z)
            task_loss = bce(logits, yb)
            adv_loss  = sum(
                bce(ah(z).squeeze(1), sv_t[i][bidx])
                for i, ah in enumerate(adv_heads)
            ) / n_attrs
            loss = task_loss + adv_weight * adv_loss
            loss.backward()
            nn.utils.clip_grad_norm_(
                list(enc.parameters()) + list(head.parameters()), 0.5)
            opt.step()
            opt_adv.step()

        sched.step()
        enc.eval(); head.eval()
        with torch.no_grad():
            pv = torch.sigmoid(head(enc(Xv))).cpu().numpy()
        auc = safe_auc(d["y_val"], pv)
        V, _, _, _, _ = audit(1, d["y_val"], pv, d["sv_val"], d["attr_names"], metric=metric)
        score = V + 0.12 * (1 - auc if not np.isnan(auc) else 0)
        if score < best_score:
            best_score = score
            best_enc   = copy.deepcopy(enc.state_dict())
            best_head  = copy.deepcopy(head.state_dict())
            no_imp     = 0
        else:
            no_imp += 1
        pbar.set_postfix({"auc": f"{auc:.4f}", "V": f"{V:.4f}", "no_imp": no_imp})

        if no_imp >= patience:
            break
    pbar.close()

    enc.load_state_dict(best_enc); head.load_state_dict(best_head)
    enc.eval(); head.eval()
    with torch.no_grad():
        proba = torch.sigmoid(head(enc(Xte))).cpu().numpy()
    tag = f"Adv-{metric.upper()} (Zhang et al., tuned)"
    return evaluate(proba, d["y_te"], d["sv_te"], d["attr_names"], tag=tag, verbose=False)


def run_fairlearn_adv(d, cfg, metric, seed=42, learning_rate=None):
    from fairlearn.adversarial import AdversarialFairnessClassifier
    set_seed(seed)
    X_tr_np = d["X_tr"].astype(np.float32)
    y_tr_np = d["y_tr"].astype(int)
    X_te_np = d["X_te"].astype(np.float32)
    y_te_np = d["y_te"].astype(int)
    sf_tr   = d["sv_tr"][0].astype(int)
    constraint = "equalized_odds" if metric == "eo" else "demographic_parity"

    uniq_sf = np.unique(sf_tr)
    if len(uniq_sf) < 2:
        raise RuntimeError(
            f"Fairlearn pre-flight check failed: sensitive feature sf_tr[0] is "
            f"single-valued ({uniq_sf.tolist()}) for metric={metric}, "
            f"ds_key={d.get('ds_key','?')}, n={len(sf_tr)}."
        )

    kwargs = dict(
        backend="torch",
        predictor_model=[128, "relu", 64, "relu"],
        adversary_model=[64, "relu"],
        constraints=constraint,
        epochs=min(cfg["epochs"], 50),
        batch_size=min(cfg["batch"], 2048),
        random_state=seed,
        progress_updates=None,
    )
    if learning_rate is not None:
        kwargs["learning_rate"] = learning_rate

    mitigator = AdversarialFairnessClassifier(**kwargs)

    _orig_bce = nn.functional.binary_cross_entropy
    def _clamped_bce(input, target, *bce_args, **bce_kwargs):
        input = torch.nan_to_num(input, nan=0.5, posinf=1 - 1e-7, neginf=1e-7)
        input = torch.clamp(input, 1e-7, 1 - 1e-7)
        return _orig_bce(input, target, *bce_args, **bce_kwargs)
    nn.functional.binary_cross_entropy = _clamped_bce

    try:
        mitigator.fit(X_tr_np, y_tr_np, sensitive_features=sf_tr)
    except Exception as e:
        tb = traceback.format_exc()
        raise RuntimeError(
            f"Fairlearn .fit() failed for metric={metric}, "
            f"ds_key={d.get('ds_key','?')}, seed={seed}, "
            f"sf_tr unique values={uniq_sf.tolist()}, n={len(sf_tr)}.\n"
            f"Original exception: {type(e).__name__}: {e}\n"
            f"Full traceback:\n{tb}"
        ) from e
    finally:
        nn.functional.binary_cross_entropy = _orig_bce

    proba = None
    for getter in ["predict_proba", "decision_function"]:
        fn = getattr(mitigator, getter, None)
        if fn is not None:
            try:
                raw = fn(X_te_np)
                raw = np.asarray(raw, dtype=np.float64)
                if raw.ndim == 2 and raw.shape[1] == 2:
                    proba = raw[:, 1]
                else:
                    proba = raw.ravel()
                if not np.all(np.isfinite(proba)):
                    proba = None
                    continue
                lo, hi = float(np.min(proba)), float(np.max(proba))
                if hi > lo:
                    proba = (proba - lo) / (hi - lo)
                else:
                    proba = np.full_like(proba, 0.5)
                break
            except Exception:
                proba = None

    if proba is None:
        try:
            hard = np.asarray(mitigator.predict(X_te_np), dtype=np.float64)
            uniq = set(np.unique(hard).tolist())
            if uniq <= {-1.0, 1.0}:
                hard = (hard + 1.0) / 2.0
            hard = np.clip(hard, 0.0, 1.0)
            proba = hard + np.random.default_rng(seed).uniform(-1e-4, 1e-4, size=hard.shape)
        except Exception as e:
            tb = traceback.format_exc()
            raise RuntimeError(
                f"Fairlearn produced no usable probability output for "
                f"metric={metric}, ds_key={d.get('ds_key','?')}, seed={seed}: "
                f"{type(e).__name__}: {e}\nFull traceback:\n{tb}"
            )

    proba = np.clip(np.nan_to_num(proba, nan=0.5, posinf=1.0, neginf=0.0), 1e-7, 1 - 1e-7)
    tag   = f"FL-AdvDeb-{metric.upper()} (tuned)"
    return evaluate(proba, y_te_np, d["sv_te"], d["attr_names"], tag=tag, verbose=False)


def _prejudice_remover_loss(prob, y, sv_list, metric, eta, eps=1e-6):
    mi_total = torch.tensor(0.0, device=prob.device)
    for sv in sv_list:
        sv = sv.float()
        p_y1 = prob.mean().clamp(eps, 1 - eps)
        p_y0 = (1 - prob).mean().clamp(eps, 1 - eps)
        p_s1 = sv.mean().clamp(eps, 1 - eps)
        p_s0 = (1 - sv).mean().clamp(eps, 1 - eps)
        p_y1_s1 = (prob * sv).mean().clamp(eps, 1 - eps)
        p_y1_s0 = (prob * (1 - sv)).mean().clamp(eps, 1 - eps)
        p_y0_s1 = ((1 - prob) * sv).mean().clamp(eps, 1 - eps)
        p_y0_s0 = ((1 - prob) * (1 - sv)).mean().clamp(eps, 1 - eps)
        if metric == "eo":
            pos_mask = (y == 1)
            if pos_mask.sum() < 2: continue
            prob_pos = prob[pos_mask]; sv_pos = sv[pos_mask]
            p_y1_s1g = (prob_pos * sv_pos).mean().clamp(eps, 1 - eps)
            p_y1_s0g = (prob_pos * (1 - sv_pos)).mean().clamp(eps, 1 - eps)
            p_s1g    = sv_pos.mean().clamp(eps, 1 - eps)
            p_s0g    = (1 - sv_pos).mean().clamp(eps, 1 - eps)
            mi_total += (
                p_y1_s1g * torch.log(p_y1_s1g / (p_y1_s1g.detach() * p_s1g + eps)) +
                p_y1_s0g * torch.log(p_y1_s0g / (p_y1_s0g.detach() * p_s0g + eps))
            )
        else:
            mi_total += (
                p_y1_s1 * torch.log(p_y1_s1 / (p_y1 * p_s1 + eps)) +
                p_y1_s0 * torch.log(p_y1_s0 / (p_y1 * p_s0 + eps)) +
                p_y0_s1 * torch.log(p_y0_s1 / (p_y0 * p_s1 + eps)) +
                p_y0_s0 * torch.log(p_y0_s0 / (p_y0 * p_s0 + eps))
            )
    return eta * mi_total / max(len(sv_list), 1)


def run_prejudice_remover(d, cfg, metric, seed=42, eta_override=None,
                           patience=PATIENCE, show_progress=True, progress_desc=""):
    set_seed(seed)
    eta       = eta_override if eta_override is not None else 2.0
    acc_floor = VANILLA_ACC[d["ds_key"]] - ACC_TOLERANCE
    Xt   = torch.tensor(d["X_tr"],  dtype=torch.float32, device=DEVICE)
    yt   = torch.tensor(d["y_tr"],  dtype=torch.float32, device=DEVICE)
    Xv   = torch.tensor(d["X_val"], dtype=torch.float32, device=DEVICE)
    Xte  = torch.tensor(d["X_te"],  dtype=torch.float32, device=DEVICE)
    sv_t = [torch.tensor(sv, dtype=torch.float32, device=DEVICE) for sv in d["sv_tr"]]
    idx_t = torch.arange(len(d["y_tr"]), dtype=torch.long)
    enc  = Encoder(d["X_tr"].shape[1]).to(DEVICE)
    head = PredictorHead(enc.rep_dim).to(DEVICE)
    bce  = nn.BCEWithLogitsLoss()
    opt  = optim.AdamW(list(enc.parameters()) + list(head.parameters()),
                       lr=LR, weight_decay=WEIGHT_DECAY)
    sched  = optim.lr_scheduler.CosineAnnealingLR(opt, cfg["epochs"], eta_min=LR * 0.01)
    loader = DataLoader(TensorDataset(Xt, yt, idx_t),
                        batch_size=cfg["batch"], shuffle=True, drop_last=True)
    best_score = np.inf; best_enc = best_head = None; no_imp = 0
    PR_ACC_PENALTY_COEF = 3.0
    pbar = tqdm(range(cfg["epochs"]), desc=progress_desc, leave=False, disable=not show_progress)
    for epoch in pbar:
        enc.train(); head.train()
        for xb, yb, bidx in loader:
            opt.zero_grad(set_to_none=True)
            z      = enc(xb)
            logits = head(z)
            prob   = torch.sigmoid(logits)
            sv_b   = [sv[bidx] for sv in sv_t]
            loss   = (bce(logits, yb)
                      + _prejudice_remover_loss(prob, yb.long(), sv_b, metric, eta))
            loss.backward()
            nn.utils.clip_grad_norm_(
                list(enc.parameters()) + list(head.parameters()), 0.5)
            opt.step()
        sched.step()
        enc.eval(); head.eval()
        with torch.no_grad():
            pv  = torch.sigmoid(head(enc(Xv))).cpu().numpy()
        acc = accuracy_score(d["y_val"], (pv > 0.5).astype(int))
        auc = safe_auc(d["y_val"], pv)
        V, _, _, _, _ = audit(1, d["y_val"], pv, d["sv_val"], d["attr_names"], metric=metric)
        score = (V
                 + 0.12 * (1 - auc if not np.isnan(auc) else 0)
                 + PR_ACC_PENALTY_COEF * max(0.0, acc_floor - acc))
        if score < best_score:
            best_score = score
            best_enc   = copy.deepcopy(enc.state_dict())
            best_head  = copy.deepcopy(head.state_dict())
            no_imp     = 0
        else:
            no_imp += 1
        pbar.set_postfix({"acc": f"{acc:.4f}", "auc": f"{auc:.4f}", "no_imp": no_imp})

        if no_imp >= patience:
            break
    pbar.close()
    enc.load_state_dict(best_enc); head.load_state_dict(best_head)
    enc.eval(); head.eval()
    with torch.no_grad():
        proba = torch.sigmoid(head(enc(Xte))).cpu().numpy()
    tag = f"PrejRem-{metric.upper()} (eta={eta:.2f}, tuned)"
    return evaluate(proba, d["y_te"], d["sv_te"], d["attr_names"], tag=tag, verbose=False)


def load_adult():
    from ucimlrepo import fetch_ucirepo
    repo  = fetch_ucirepo(id=2)
    X_df  = repo.data.features.copy()
    y_ser = repo.data.targets.copy()
    y_raw = y_ser.iloc[:, 0].astype(str).str.strip().str.rstrip(".")
    y     = (y_raw == ">50K").astype(int).values
    race_Black = (X_df["race"].astype(str).str.strip() == "Black").astype(int).values
    race_White = (X_df["race"].astype(str).str.strip() == "White").astype(int).values
    sex_Female = (X_df["sex"].astype(str).str.strip() == "Female").astype(int).values
    age_old    = (pd.to_numeric(X_df["age"], errors="coerce").fillna(0) >= 45).astype(int).values
    X_df = X_df.drop(columns=["race","sex","age","fnlwgt","education-num"], errors="ignore")
    X_df = pd.get_dummies(X_df)
    X    = X_df.values.astype(np.float32)
    valid = ~np.isnan(X).any(axis=1)
    X, y  = X[valid], y[valid]
    race_Black=race_Black[valid]; race_White=race_White[valid]
    sex_Female=sex_Female[valid]; age_old=age_old[valid]
    tr, te = strat_split(X, y, [race_Black, sex_Female, age_old])
    sc = StandardScaler()
    X_tr = sc.fit_transform(X[tr]); X_te = sc.transform(X[te])
    attr_names = ["race_Black","race_White","sex_Female","age_old"]
    sv_tr = [race_Black[tr], race_White[tr], sex_Female[tr], age_old[tr]]
    sv_te = [race_Black[te], race_White[te], sex_Female[te], age_old[te]]
    tr2, val = strat_split(X_tr, y[tr], sv_tr)
    return dict(ds_key="adult",
        X_tr=X_tr[tr2], y_tr=y[tr][tr2], X_val=X_tr[val], y_val=y[tr][val],
        X_te=X_te, y_te=y[te],
        sv_tr=[ensure_binary(sv[tr2]) for sv in sv_tr],
        sv_val=[ensure_binary(sv[val]) for sv in sv_tr],
        sv_te=[ensure_binary(sv) for sv in sv_te],
        attr_names=attr_names)

def load_acs_income():
    from folktables import ACSDataSource, ACSIncome
    ds  = ACSDataSource(survey_year="2018", horizon="1-Year", survey="person")
    acs = ds.get_data(states=["CA"], download=True)
    features, label, group = ACSIncome.df_to_numpy(acs)
    acs_feature_cols = ["AGEP","COW","SCHL","MAR","OCCP","POBP","RELP","WKHP","SEX","RAC1P"]
    sex_idx  = acs_feature_cols.index("SEX"); race_idx = acs_feature_cols.index("RAC1P")
    age_idx  = acs_feature_cols.index("AGEP")
    sex_col  = features[:, sex_idx]; race_col = features[:, race_idx]; age_col = features[:, age_idx]
    rW=(race_col==1).astype(int); rB=(race_col==2).astype(int)
    rA=(race_col==6).astype(int); sex=(sex_col==2).astype(int); age_old=(age_col>=45).astype(int)
    valid    = ~np.isnan(label.astype(float))
    features = features[valid].astype(np.float32); label = label[valid].astype(int)
    rW=rW[valid]; rB=rB[valid]; rA=rA[valid]; sex=sex[valid]; age_old=age_old[valid]
    tr, te = strat_split(features, label, [rW, rB, sex, age_old])
    sc = StandardScaler()
    X_tr = sc.fit_transform(features[tr]); X_te = sc.transform(features[te])
    sv_tr = [rW[tr], rB[tr], rA[tr], sex[tr], age_old[tr]]
    sv_te = [rW[te], rB[te], rA[te], sex[te], age_old[te]]
    attr_names = ["race_White","race_Black","race_Asian","sex_Female","age_old"]
    tr2, val = strat_split(X_tr, label[tr], sv_tr)
    return dict(ds_key="acs_income",
        X_tr=X_tr[tr2], y_tr=label[tr][tr2], X_val=X_tr[val], y_val=label[tr][val],
        X_te=X_te, y_te=label[te],
        sv_tr=[ensure_binary(sv[tr2]) for sv in sv_tr],
        sv_val=[ensure_binary(sv[val]) for sv in sv_tr],
        sv_te=[ensure_binary(sv) for sv in sv_te],
        attr_names=attr_names)

def load_acs_employment():
    from folktables import ACSDataSource, ACSEmployment
    ds  = ACSDataSource(survey_year="2018", horizon="1-Year", survey="person")
    acs = ds.get_data(states=["CA"], download=True)
    features, label, _ = ACSEmployment.df_to_numpy(acs)
    acs_emp_cols = ["AGEP","SCHL","MAR","DIS","ESP","CIT","MIG","MIL","ANC",
                    "NATIVITY","DEAR","DEYE","DREM","SEX","RAC1P"]
    sex_idx  = acs_emp_cols.index("SEX"); race_idx = acs_emp_cols.index("RAC1P")
    age_idx  = acs_emp_cols.index("AGEP"); dis_idx  = acs_emp_cols.index("DIS")
    sex_col  = features[:, sex_idx]; race_col = features[:, race_idx]
    age_col  = features[:, age_idx]; dis_col  = features[:, dis_idx]
    RACE_MAP = {1:0, 2:1, 3:3, 4:2, 5:2, 6:3, 7:3, 8:3, 9:3}
    race = np.array([RACE_MAP.get(int(r), 3) for r in race_col])
    sex  = (sex_col == 2).astype(int)
    rW=(race==0).astype(int); rB=(race==1).astype(int); rO=(race==3).astype(int)
    age_old=(age_col>=45).astype(int); disabled=(dis_col==1).astype(int)
    valid    = ~np.isnan(label.astype(float))
    features = features[valid].astype(np.float32); label = label[valid].astype(int)
    rW=rW[valid]; rB=rB[valid]; rO=rO[valid]; sex=sex[valid]
    age_old=age_old[valid]; disabled=disabled[valid]
    tr, te = strat_split(features, label, [rW, rB, sex, age_old, disabled])
    sc = StandardScaler()
    X_tr = sc.fit_transform(features[tr]); X_te = sc.transform(features[te])
    sv_tr = [rW[tr], rB[tr], rO[tr], sex[tr], age_old[tr], disabled[tr]]
    sv_te = [rW[te], rB[te], rO[te], sex[te], age_old[te], disabled[te]]
    attr_names = ["race_White","race_Black","race_Other","sex_Female","age_old","disabled"]
    tr2, val = strat_split(X_tr, label[tr], sv_tr)
    return dict(ds_key="acs_employment",
        X_tr=X_tr[tr2], y_tr=label[tr][tr2], X_val=X_tr[val], y_val=label[tr][val],
        X_te=X_te, y_te=label[te],
        sv_tr=[ensure_binary(sv[tr2]) for sv in sv_tr],
        sv_val=[ensure_binary(sv[val]) for sv in sv_tr],
        sv_te=[ensure_binary(sv) for sv in sv_te],
        attr_names=attr_names)

def load_communities_crime():
    from ucimlrepo import fetch_ucirepo
    repo = fetch_ucirepo(id=183)
    X_df = repo.data.features.copy()
    y_df = repo.data.targets.copy()
    y_cont = pd.to_numeric(y_df.iloc[:, 0], errors="coerce").values
    valid  = ~np.isnan(y_cont)
    y_cont = y_cont[valid]; X_df = X_df[valid].reset_index(drop=True)
    y = (y_cont > np.median(y_cont)).astype(int)
    def find_col(df, pats):
        for p in pats:
            m = [c for c in df.columns if p.lower() in c.lower()]
            if m: return pd.to_numeric(df[m[0]], errors="coerce")
        return None
    col_b = find_col(X_df, ["racepctblack","pctblack"])
    col_i = find_col(X_df, ["medIncome","medincome"])
    def binarise(col, high=True):
        if col is None: return np.zeros(len(y), int)
        c = col.fillna(col.median()).values
        return (c > np.median(c)).astype(int) if high else (c < np.median(c)).astype(int)
    s_race = binarise(col_b, high=True); s_inc = binarise(col_i, high=False)
    X_num = X_df.apply(pd.to_numeric, errors="coerce")
    X_num = X_num.dropna(axis=1, thresh=int(0.7*len(X_num))).fillna(X_num.median())
    drop_cols = [c for c in X_num.columns if any(p.lower() in c.lower()
                 for p in ["racepct","racePct","medIncome","ViolentCrimes","PctUnemployed"])]
    X = X_num.drop(columns=drop_cols, errors="ignore").values.astype(np.float32)
    tr, te = strat_split(X, y, [s_race, s_inc])
    sc = StandardScaler()
    X_tr = sc.fit_transform(X[tr]); X_te = sc.transform(X[te])
    sv_tr = [s_race[tr], s_inc[tr]]; sv_te = [s_race[te], s_inc[te]]
    attr_names = ["racial_composition","socioeconomic"]
    tr2, val = strat_split(X_tr, y[tr], sv_tr)
    return dict(ds_key="communities_crime",
        X_tr=X_tr[tr2], y_tr=y[tr][tr2], X_val=X_tr[val], y_val=y[tr][val],
        X_te=X_te, y_te=y[te],
        sv_tr=[ensure_binary(sv[tr2]) for sv in sv_tr],
        sv_val=[ensure_binary(sv[val]) for sv in sv_tr],
        sv_te=[ensure_binary(sv) for sv in sv_te],
        attr_names=attr_names)

LOADERS = {
    "adult"            : load_adult,
    "acs_income"       : load_acs_income,
    "acs_employment"   : load_acs_employment,
    "communities_crime": load_communities_crime,
}


# ---- outer progress bar bookkeeping ----
# vanilla(1) + [adv, fairlearn, prejudice_rem, agad, no_auditor](5) * 2 metrics = 11 method-slots per dataset
METHODS_PER_DATASET = 1 + 5 * 2
TOTAL_METHOD_RUNS = METHODS_PER_DATASET * len(RUN_DATASETS) * len(SEEDS_FINAL)

t_phase = time.time()
all_results = {}
all_best_params = {}

outer_pbar = tqdm(total=TOTAL_METHOD_RUNS, desc="TOTAL PROGRESS", position=0)

for ds_key in RUN_DATASETS:
    print_subsection(f"Dataset: {ds_key.upper()}")
    cfg = FULL_CFG[ds_key]
    d   = LOADERS[ds_key]()
    print(f"  Train={len(d['y_tr'])}  Val={len(d['y_val'])}  Test={len(d['y_te'])}")

    ds_results = {}
    ds_best_params = HARDCODED_PARAMS[ds_key]

    print(f"\n  -- vanilla --")
    seed_results = []
    for s in SEEDS_FINAL:
        t0 = time.time()
        r = run_vanilla(d, cfg, seed=s, progress_desc=f"{ds_key}/vanilla/seed{s}")
        dt = time.time() - t0
        print(f"    seed={s:<5} acc={r['acc']:.4f} auc={r['auc']:.4f} "
              f"wc_eo={r['wc_eo']:.4f} wc_dp={r['wc_dp']:.4f}  ({dt:.1f}s)")
        seed_results.append(r)
        outer_pbar.update(1)
    ds_results["vanilla"] = aggregate_seeds(seed_results, tag_for_warning=f"vanilla/{ds_key}")
    print(f"    >> vanilla/{ds_key} agg: acc={ds_results['vanilla']['acc']:.4f}\u00b1{ds_results['vanilla']['acc_std']:.4f}  "
          f"auc={ds_results['vanilla']['auc']:.4f}\u00b1{ds_results['vanilla']['auc_std']:.4f}")

    for metric in ["eo", "dp"]:
        print(f"\n  === {metric.upper()} ===")

        p = ds_best_params[f"adv_{metric}"]
        print(f"  -- adv_{metric} (hardcoded params={p}) --")
        seed_results = []
        for s in SEEDS_FINAL:
            t0 = time.time()
            r = _adv_static(d, cfg, metric, seed=s, patience=PATIENCE_FINAL,
                             progress_desc=f"{ds_key}/adv_{metric}/seed{s}", **p)
            dt = time.time() - t0
            print(f"    seed={s:<5} acc={r['acc']:.4f} auc={r['auc']:.4f} "
                  f"wc_{metric}={r[f'wc_{metric}']:.4f}  ({dt:.1f}s)")
            seed_results.append(r)
            outer_pbar.update(1)
        ds_results[f"adv_{metric}"] = aggregate_seeds(seed_results, tag_for_warning=f"adv_{metric}/{ds_key}")
        agg = ds_results[f"adv_{metric}"]
        print(f"    >> adv_{metric}/{ds_key} agg: acc={agg['acc']:.4f}\u00b1{agg['acc_std']:.4f}  "
              f"wc_{metric}={agg[f'wc_{metric}']:.4f}\u00b1{agg[f'wc_{metric}_std']:.4f}")

        p = ds_best_params[f"fairlearn_{metric}"]
        print(f"  -- fairlearn_{metric} (hardcoded params={p}) --")
        seed_results = []
        for s in tqdm(SEEDS_FINAL, desc=f"{ds_key}/fairlearn_{metric}", leave=False):
            t0 = time.time()
            try:
                r = run_fairlearn_adv(d, cfg, metric, seed=s, **p)
                dt = time.time() - t0
                print(f"    seed={s:<5} acc={r['acc']:.4f} auc={r['auc']:.4f} "
                      f"wc_{metric}={r[f'wc_{metric}']:.4f}  ({dt:.1f}s)")
            except Exception as e:
                print(f"    [fairlearn seed {s} failed: {e}]")
                r = {k: float("nan") for k in ["wc_eo","wc_dp","fg_eo","fg_dp","acc","auc"]}
            seed_results.append(r)
            outer_pbar.update(1)
        ds_results[f"fairlearn_{metric}"] = aggregate_seeds(seed_results, tag_for_warning=f"fairlearn_{metric}/{ds_key}")
        agg = ds_results[f"fairlearn_{metric}"]
        print(f"    >> fairlearn_{metric}/{ds_key} agg: acc={agg['acc']:.4f}\u00b1{agg['acc_std']:.4f}  "
              f"wc_{metric}={agg[f'wc_{metric}']:.4f}\u00b1{agg[f'wc_{metric}_std']:.4f}")

        p = ds_best_params[f"prejudice_rem_{metric}"]
        print(f"  -- prejudice_rem_{metric} (hardcoded params={p}) --")
        seed_results = []
        for s in SEEDS_FINAL:
            t0 = time.time()
            r = run_prejudice_remover(d, cfg, metric, seed=s, eta_override=p["eta"],
                                       patience=PATIENCE_FINAL,
                                       progress_desc=f"{ds_key}/prejrem_{metric}/seed{s}")
            dt = time.time() - t0
            print(f"    seed={s:<5} acc={r['acc']:.4f} auc={r['auc']:.4f} "
                  f"wc_{metric}={r[f'wc_{metric}']:.4f}  ({dt:.1f}s)")
            seed_results.append(r)
            outer_pbar.update(1)
        ds_results[f"prejudice_rem_{metric}"] = aggregate_seeds(seed_results, tag_for_warning=f"prejrem_{metric}/{ds_key}")
        agg = ds_results[f"prejudice_rem_{metric}"]
        print(f"    >> prejudice_rem_{metric}/{ds_key} agg: acc={agg['acc']:.4f}\u00b1{agg['acc_std']:.4f}  "
              f"wc_{metric}={agg[f'wc_{metric}']:.4f}\u00b1{agg[f'wc_{metric}_std']:.4f}")

        p = ds_best_params[f"agad_{metric}"]
        print(f"  -- agad_{metric} (hardcoded params={p}) --")
        seed_results = []
        for s in SEEDS_FINAL:
            t0 = time.time()
            r = run_agad_with_hparams(d, cfg, metric, p, seed=s,
                                       verbose=False, epoch_verbose=False,
                                       disable_auditor=False,
                                       patience=PATIENCE_FINAL,
                                       progress_desc=f"{ds_key}/agad_{metric}/seed{s}")
            dt = time.time() - t0
            print(f"    seed={s:<5} acc={r['acc']:.4f} auc={r['auc']:.4f} "
                  f"wc_{metric}={r[f'wc_{metric}']:.4f}  best_epoch={r['best_epoch']}  ({dt:.1f}s)")
            seed_results.append(r)
            outer_pbar.update(1)
        ds_results[f"agad_{metric}_tuned"] = aggregate_seeds(seed_results, tag_for_warning=f"agad_{metric}/{ds_key}")
        agg = ds_results[f"agad_{metric}_tuned"]
        print(f"    >> agad_{metric}/{ds_key} agg: acc={agg['acc']:.4f}\u00b1{agg['acc_std']:.4f}  "
              f"wc_{metric}={agg[f'wc_{metric}']:.4f}\u00b1{agg[f'wc_{metric}_std']:.4f}")

        seed_results = []
        for s in SEEDS_FINAL:
            t0 = time.time()
            r = run_agad_with_hparams(d, cfg, metric, p, seed=s,
                                       verbose=False, epoch_verbose=False,
                                       disable_auditor=True,
                                       patience=PATIENCE_FINAL,
                                       progress_desc=f"{ds_key}/no_auditor_{metric}/seed{s}")
            dt = time.time() - t0
            print(f"    [no_auditor] seed={s:<5} acc={r['acc']:.4f} auc={r['auc']:.4f} "
                  f"wc_{metric}={r[f'wc_{metric}']:.4f}  ({dt:.1f}s)")
            seed_results.append(r)
            outer_pbar.update(1)
        ds_results[f"abl_no_auditor_{metric}"] = aggregate_seeds(seed_results, tag_for_warning=f"no_auditor_{metric}/{ds_key}")
        agg = ds_results[f"abl_no_auditor_{metric}"]
        print(f"    >> no_auditor_{metric}/{ds_key} agg: acc={agg['acc']:.4f}\u00b1{agg['acc_std']:.4f}  "
              f"wc_{metric}={agg[f'wc_{metric}']:.4f}\u00b1{agg[f'wc_{metric}_std']:.4f}")

    all_results[ds_key] = ds_results
    all_best_params[ds_key] = ds_best_params

    elapsed_so_far = time.time() - t_phase
    print(f"\n  [TIME] elapsed so far: {elapsed_so_far/60:.1f} min "
          f"({outer_pbar.n}/{outer_pbar.total} method-runs done)")

    print(f"\n  -- HONEST comparison summary for {ds_key} --")
    acc_floor_ds = VANILLA_ACC[ds_key] - BASELINE_ACC_FLOOR_TOLERANCE
    auc_floor_ds = VANILLA_AUC[ds_key] - BASELINE_AUC_FLOOR_TOLERANCE

    def _paired_verdict(a_vals, b_vals, label_a, label_b):
        a_vals = [v for v in a_vals if v is not None and not (isinstance(v, float) and np.isnan(v))]
        b_vals = [v for v in b_vals if v is not None and not (isinstance(v, float) and np.isnan(v))]
        if len(a_vals) < 2 or len(b_vals) < 2 or len(a_vals) != len(b_vals):
            return f"{label_a} vs {label_b}: insufficient paired seeds for a significance test"
        t, p = stats.ttest_rel(a_vals, b_vals)
        sig = "significant (p<0.05)" if p < 0.05 else "NOT significant"
        return (f"{label_a}={np.mean(a_vals):.4f} vs {label_b}={np.mean(b_vals):.4f}, "
                f"paired t-test p={p:.3f} [{sig}]")

    for metric in ["eo", "dp"]:
        wc_key = f"wc_{metric}"
        all_candidates = {
            "adv"           : ds_results[f"adv_{metric}"],
            "fairlearn"     : ds_results[f"fairlearn_{metric}"],
            "prejudice_rem" : ds_results[f"prejudice_rem_{metric}"],
            "agad"          : ds_results[f"agad_{metric}_tuned"],
            "no_auditor"    : ds_results[f"abl_no_auditor_{metric}"],
        }

        qualified, disqualified = {}, []
        for name, r in all_candidates.items():
            acc = r.get("acc", float("nan"))
            auc = r.get("auc", float("nan"))
            wc  = r.get(wc_key, float("nan"))
            if np.isnan(acc) or np.isnan(wc):
                disqualified.append(f"{name}(missing/failed run)")
                continue
            reasons = []
            if acc < acc_floor_ds:
                reasons.append(f"acc={acc:.4f}<{acc_floor_ds:.4f}")
            # AUC floor disqualification removed (USE_AUC_FLOOR=False by default).
            # AUC is still reported below for diagnostic purposes.
            if USE_AUC_FLOOR and not np.isnan(auc) and auc < auc_floor_ds:
                reasons.append(f"auc={auc:.4f}<{auc_floor_ds:.4f}")
            if reasons:
                disqualified.append(f"{name}({', '.join(reasons)})")
            else:
                qualified[name] = wc

        agad_qualified = "agad" in qualified
        baseline_only  = {k: v for k, v in qualified.items() if k not in ("agad", "no_auditor")}
        best_baseline      = min(baseline_only.values()) if baseline_only else float("nan")
        best_baseline_name = min(baseline_only, key=baseline_only.get) if baseline_only else "none"

        if not agad_qualified:
            verdict = "AGAD FAILS ITS OWN ACC FLOOR -- not a valid comparison point this run"
        elif np.isnan(best_baseline):
            verdict = "AGAD best (no qualified baseline)"
        elif qualified["agad"] <= best_baseline:
            verdict = "AGAD best"
        else:
            verdict = "AGAD does NOT beat best (acc-floor-qualified) baseline"

        auditor_note = "auditor comparison unavailable (agad or no_auditor disqualified/failed)"
        if "agad" in qualified and "no_auditor" in qualified:
            auditor_note = "auditor helps" if qualified["agad"] < qualified["no_auditor"] else "auditor does NOT help here"

        disq_str = f"  [disqualified: {', '.join(disqualified)}]" if disqualified else ""
        full_str = f"{ds_results[f'agad_{metric}_tuned'][wc_key]:.4f}"
        abl_str  = f"{ds_results[f'abl_no_auditor_{metric}'][wc_key]:.4f}"
        best_str = f"{best_baseline:.4f}" if not np.isnan(best_baseline) else "n/a"
        print(f"    {metric.upper()}: AGAD={full_str}  best_qualified_baseline={best_str} "
              f"({best_baseline_name})  [{verdict}]  |  no_auditor={abl_str}  [{auditor_note}]{disq_str}")
        # Report AUC for all candidates regardless of qualification, purely diagnostic
        auc_report = "  ".join(f"{name}_auc={r.get('auc', float('nan')):.4f}"
                                for name, r in all_candidates.items())
        print(f"      [auc, diagnostic only, not gating] {auc_report}")

        agad_all = ds_results[f"agad_{metric}_tuned"].get(f"{wc_key}_all", [])
        abl_all  = ds_results[f"abl_no_auditor_{metric}"].get(f"{wc_key}_all", [])
        print(f"      [sig] AGAD vs no-auditor: "
              f"{_paired_verdict(agad_all, abl_all, 'AGAD', 'no-auditor')}")
        if best_baseline_name != "none":
            base_all = all_candidates[best_baseline_name].get(f"{wc_key}_all", [])
            print(f"      [sig] AGAD vs best baseline ({best_baseline_name}): "
                  f"{_paired_verdict(agad_all, base_all, 'AGAD', best_baseline_name)}")

    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

outer_pbar.close()
run_time = time.time() - t_phase
print(f"\n[TIME] Total main phase runtime: {run_time/60:.1f} min")


def track_worst_subgroup(ds_key, metric, hp, seed=42, epoch_cap=60):
    cfg = FULL_CFG[ds_key]
    d = LOADERS[ds_key]()
    set_seed(seed)

    Xt = torch.tensor(d["X_tr"], dtype=torch.float32, device=DEVICE)
    yt = torch.tensor(d["y_tr"], dtype=torch.float32, device=DEVICE)
    Xv = torch.tensor(d["X_val"], dtype=torch.float32, device=DEVICE)

    enc = Encoder(d["X_tr"].shape[1]).to(DEVICE)
    head = PredictorHead(enc.rep_dim).to(DEVICE)
    opt = optim.AdamW(list(enc.parameters()) + list(head.parameters()), lr=1e-3)
    loader = DataLoader(TensorDataset(Xt, yt), batch_size=cfg["batch"],
                         shuffle=True, drop_last=True)
    bce = nn.BCEWithLogitsLoss()

    log = []
    for epoch in tqdm(range(min(cfg["epochs"], epoch_cap)),
                       desc=f"worst-subgroup-track/{ds_key}/{metric}", leave=False):
        enc.eval(); head.eval()
        with torch.no_grad():
            pv = torch.sigmoid(head(enc(Xv))).cpu().numpy()
        worst_gap, worst_spec, fg_k, _, _ = audit(1, d["y_val"], pv, d["sv_val"],
                                                    d["attr_names"], metric=metric)
        label = "+".join(f"{a}={v}" for a, v in worst_spec) if worst_spec else "none"
        log.append((epoch, label, worst_gap))

        enc.train(); head.train()
        for xb, yb in loader:
            opt.zero_grad(set_to_none=True)
            bce(head(enc(xb)), yb).backward()
            opt.step()

    return log


def plot_worst_subgroup_timeline(log, ds_key, metric, outpath):
    epochs = [e for e, _, _ in log]
    labels = [l for _, l, _ in log]
    gaps   = [g for _, _, g in log]
    unique_labels = sorted(set(labels))
    label_to_y = {l: i for i, l in enumerate(unique_labels)}
    ys = [label_to_y[l] for l in labels]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True,
                                    gridspec_kw={"height_ratios": [2, 1]})
    ax1.step(epochs, ys, where="post", color="#1a7c34", linewidth=1.5)
    ax1.scatter(epochs, ys, color="#1a7c34", s=15, zorder=3)
    ax1.set_yticks(range(len(unique_labels)))
    ax1.set_yticklabels(unique_labels, fontsize=7)
    ax1.set_ylabel("Worst-performing subgroup")
    ax1.set_title(f"Worst-Subgroup Identity Over Training \u2014 {DS_LABELS[ds_key]} ({metric.upper()})\n"
                   f"{len(unique_labels)} distinct subgroups held the 'worst' position "
                   f"across {len(epochs)} checkpoints")
    ax1.grid(alpha=0.3)
    ax2.plot(epochs, gaps, color="#c00000", linewidth=1.5)
    ax2.set_xlabel("Epoch"); ax2.set_ylabel(f"Worst-case {metric.upper()} gap")
    ax2.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(outpath, dpi=150, bbox_inches="tight")
    plt.close()
    n_switches = sum(1 for i in range(1, len(labels)) if labels[i] != labels[i-1])
    print(f"  {ds_key}/{metric}: {n_switches} switches in worst-subgroup identity "
          f"across {len(epochs)} checkpoints ({len(unique_labels)} unique subgroups)")
    print(f"  Saved -> {outpath}")


print_section("DIAGNOSTIC: Worst-Subgroup-Over-Time (non-stationarity evidence)")
for ds_key in ["acs_employment", "adult"]:
    hp = all_best_params[ds_key]["agad_eo"]
    log = track_worst_subgroup(ds_key, "eo", hp, seed=42)
    plot_worst_subgroup_timeline(
        log, ds_key, "eo",
        f"{PLOT_DIR}/{ds_key}_eo_worst_subgroup_timeline.png"
    )


print_section("GLOBAL SUMMARY")

row_order_global = [
    "vanilla", "adv_eo", "adv_dp",
    "fairlearn_eo", "fairlearn_dp",
    "prejudice_rem_eo", "prejudice_rem_dp",
    "abl_no_auditor_eo", "abl_no_auditor_dp",
    "agad_eo_tuned", "agad_dp_tuned",
]

def fmt_cell(r, k, w=18):
    v = r.get(k, float("nan")); s = r.get(k+"_std", 0.0)
    cell = f"{v:.4f}\u00b1{s:.4f}" if not np.isnan(v) else "nan"
    return f"{cell:>{w}}"

for metric_key, label in [("wc_eo", "EO"), ("wc_dp", "DP")]:
    fg_key = metric_key.replace("wc_", "fg_")
    hdr = f"  {'Dataset':<22}  {'Method':<28}  {label+'(WC)':>18}  {label+'(FG)':>18}  {'Acc':>12}  {'AUC':>12}  {'Fail':>5}"
    sep = "  " + "\u2500" * 106
    print(hdr); print(sep)
    for ds_key in RUN_DATASETS:
        rows = [m for m in row_order_global if m in all_results[ds_key]]
        for i, m in enumerate(rows):
            r = all_results[ds_key][m]
            ds_label = DS_LABELS[ds_key] if i == 0 else ""
            flag = " \u2190\u2605" if "agad" in m else ""
            n_fail = r.get("n_seeds_failed", 0)
            fail_str = f"{n_fail}/{r.get('n_seeds_total', len(SEEDS_FINAL))}" if n_fail else ""
            print(f"  {ds_label:<22}  {METHOD_LABELS.get(m,m):<28}"
                  f"{fmt_cell(r, metric_key)}{fmt_cell(r, fg_key)}"
                  f"{fmt_cell(r,'acc',12)}{fmt_cell(r,'auc',12)}  {fail_str:>5}{flag}")
        print(sep)


print_section("AUDITOR CONTRIBUTION SUMMARY")
for ds_key in RUN_DATASETS:
    print(f"  {DS_LABELS[ds_key]}:")
    for metric in ["eo", "dp"]:
        full_wc = all_results[ds_key][f"agad_{metric}_tuned"][f"wc_{metric}"]
        abl_wc  = all_results[ds_key][f"abl_no_auditor_{metric}"][f"wc_{metric}"]
        delta   = full_wc - abl_wc
        verdict = "AUDITOR HELPS" if delta < 0 else "auditor neutral/hurts (reported as-is)"
        print(f"    {metric.upper()}  full={full_wc:.4f}  no_auditor={abl_wc:.4f}  "
              f"delta={delta:+.5f}  [{verdict}]")
    print()


def clean(obj):
    if isinstance(obj, dict):          return {k: clean(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)): return [clean(v) for v in obj]
    if isinstance(obj, float) and (np.isnan(obj) or np.isinf(obj)): return None
    if isinstance(obj, (np.floating, np.integer)): return obj.item()
    if isinstance(obj, np.ndarray):   return obj.tolist()
    return obj

payload = {
    "results"            : all_results,
    "best_params_found"  : all_best_params,
    "runtime"            : {"total_min": run_time / 60},
    "seeds"              : SEEDS_FINAL,
    "hyperparam_source"  : "hardcoded from v7 Optuna results, no re-tuning performed",
    "size_weighting"     : {
        "enabled": SIZE_WEIGHT_ENABLED,
        "ref_n": SIZE_WEIGHT_REF_N,
        "min_weight": SIZE_WEIGHT_MIN,
        "note": ("Subgroup gaps used for top-k selection, fg_k, and per-subgroup "
                 "lambda are discounted by sqrt(n/ref_n) clipped to [min_weight, 1.0] "
                 "before ranking, so small/noisy subgroups need a larger raw gap to "
                 "dominate the training signal. Raw (unweighted) gaps are still used "
                 "for all reported metrics (wc_eo, wc_dp, fg_eo, fg_dp)."),
    },
    "auc_floor"          : {
        "enabled": USE_AUC_FLOOR,
        "note": "AUC is computed and reported for every method/cell but no longer "
                "disqualifies a method from being 'best' or 'qualified'. Only the "
                "accuracy floor gates qualification now.",
    },
    "fixes_applied": [
        "Fairlearn adversarial BCE clamp now applies nan_to_num before clamp, "
        "preventing NaN from surviving the clamp and crashing training.",
        "AGAD checkpoint selection now scores epochs by fg_k (mean of top-5 "
        "subgroup gaps) instead of V_t (single worst-subgroup gap), reducing "
        "sensitivity to single sparse-subgroup noise.",
        "All Optuna tuning removed; hyperparameters hardcoded from the prior "
        "20-trial search per dataset/metric/method (see HARDCODED_PARAMS).",
        "Audit ranking now size-weights each subgroup's gap by sqrt(n/ref_n) "
        "before selecting top-k / computing fg_k / setting per-subgroup lambda, "
        "so small noisy subgroups need much larger gaps to dominate training.",
        "AUC floor disqualification removed from the honest-comparison verdict "
        "logic; AUC is still computed and printed per method for diagnostic use.",
    ],
    "protocol_note"      : (
        "Hyperparameters were not re-tuned; they are hardcoded from the prior "
        "Optuna search. Checkpoint selection was changed to use fg_k, and the "
        "Fairlearn NaN-before-clamp bug was fixed. Subgroup audit ranking is now "
        "size-weighted. AUC is diagnostic only and no longer gates qualification. "
        "The final 5-seed evaluation for every method used the full dataset, "
        "full epoch budget, and full patience."
    ),
}

out_path = f"{WORK_DIR}/agad_v9_results.json"
with open(out_path, "w") as f:
    json.dump(clean(payload), f, indent=2)

print_section("DONE")
print(f"  Total time   : {run_time/60:.1f} min")
print(f"  Results      -> {out_path}")
print(f"  Plots        -> {PLOT_DIR}/")
print(f"  Protocol     : hardcoded hyperparams, no Optuna, fg_k checkpointing, "
      f"fairlearn nan fix, size-weighted audit, AUC floor removed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.5/135.5 kB 3.2 MB/s eta 0:00:00
  AGAD v9 -- hardcoded hyperparams, fg_k checkpointing, fairlearn nan fix,
             size-weighted audit, AUC floor removed, verbose + tqdm
  Device       : cuda
  Seeds        : [42, 7, 123, 2024, 99]
  TOP_K_OPT    : 5
  Per-sg lambda: [0.25, 5.0]
  Size weight  : enabled=True  ref_n=500  min=0.15
  AUC floor    : enabled=False (disqualification now driven by acc floor only)
  Curriculum   : convergence-based (threshold=0.08, patience=3)


TOTAL PROGRESS:   0%|          | 0/220 [00:00<?, ?it/s]


----------------------------------------------------------------------
  Dataset: ADULT
----------------------------------------------------------------------
  Train=31258  Val=7815  Test=9769

  -- vanilla --


adult/vanilla/seed42:   0%|          | 0/80 [00:00<?, ?it/s]

    seed=42    acc=0.7413 auc=0.8481 wc_eo=0.0762 wc_dp=0.0613  (23.3s)


...

    [no_auditor] seed=99    acc=0.8020 auc=0.9070 wc_dp=0.2740  (3.8s)
    >> no_auditor_dp/communities_crime agg: acc=0.8110±0.0092  wc_dp=0.2813±0.0103

  [TIME] elapsed so far: 484.0 min (220/220 method-runs done)

  -- HONEST comparison summary for communities_crime --
    EO: AGAD=0.0117  best_qualified_baseline=0.0440 (adv)  [AGAD best]  |  no_auditor=0.2680  [auditor helps]
      [auc, diagnostic only, not gating] adv_auc=0.8353  fairlearn_auc=0.7859  prejudice_rem_auc=0.8662  agad_auc=0.8167  no_auditor_auc=0.9051
      [sig] AGAD vs no-auditor: AGAD=0.0117 vs no-auditor=0.2680, paired t-test p=0.000 [significant (p<0.05)]
      [sig] AGAD vs best baseline (adv): AGAD=0.0117 vs adv=0.0440, paired t-test p=0.000 [significant (p<0.05)]
    DP: AGAD=0.0170  best_qualified_baseline=0.0404 (adv)  [AGAD best]  |  no_auditor=0.2813  [auditor helps]
      [auc, diagnostic only, not gating] adv_auc=0.8351  fairlearn_auc=0.7636  prejudice_rem_auc=0.7863  agad_auc=0.7405  no_auditor_auc=0

worst-subgroup-track/acs_employment/eo:   0%|          | 0/60 [00:00<?, ?it/s]

  acs_employment/eo: 0 switches in worst-subgroup identity across 60 checkpoints (1 unique subgroups)
  Saved -> /kaggle/working/agad_plots_fair/acs_employment_eo_worst_subgroup_timeline.png


worst-subgroup-track/adult/eo:   0%|          | 0/60 [00:00<?, ?it/s]

  adult/eo: 2 switches in worst-subgroup identity across 60 checkpoints (3 unique subgroups)
  Saved -> /kaggle/working/agad_plots_fair/adult_eo_worst_subgroup_timeline.png

  GLOBAL SUMMARY
  Dataset                 Method                                    EO(WC)              EO(FG)           Acc           AUC   Fail
  ──────────────────────────────────────────────────────────────────────────────────────────────────────────
  Adult Income            Vanilla NN                       0.0746±0.0134     0.0685±0.01360.7902±0.03540.8490±0.0070       
                          Adv-EO (tuned)                   0.0831±0.0289     0.0689±0.01290.8072±0.03750.8594±0.0178       
                          Adv-DP (tuned)                   0.0760±0.0134     0.0709±0.01450.7878±0.03520.8477±0.0072       
                          FL-AdvDeb-EO (tuned)             0.0993±0.0000     0.0930±0.00000.8477±0.00000.7672±0.0014       
                          FL-AdvDeb-DP (tuned)             0.0711±0.0000  